In [ ]:
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

# Load credentials from your .env file
load_dotenv()

class KnowledgeGraphReader:
    def __init__(self):
        uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
        user = os.getenv("NEO4J_USER", "neo4j")
        password = os.getenv("NEO4J_PASSWORD", "password")
        
        # Establish the connection to the existing database
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        print("✅ Successfully connected to the existing Knowledge Graph!")

    def close(self):
        self.driver.close()

    def test_connection_get_freelancers(self):
        """
        A simple query to prove we are reading the existing data.
        """
        query = """
        MATCH (f:Freelancer)
        RETURN f.name AS name
        """
        with self.driver.session() as session:
            result = session.run(query)
            freelancers = [record["name"] for record in result]
            return freelancers

    # Notice how this is indented to be INSIDE the KnowledgeGraphReader class!
    def form_optimal_teams(self, limit=10):
        """
        Executes the advanced team formation Cypher query to find the best 
        combinations of AI Engineers, UI/UX Designers, and Web Developers 
        using technical scoring and a decimal synergy tie-breaker.
        """
        # Notice the 'r' before the quotes. This is REQUIRED for the regex \b to work!
        query = r"""
        MATCH (f:Freelancer)

        // 1. DYNAMICALLY SCORE EVERY FREELANCER
        WITH f,
             size([(f)-[:HAS_SKILL]->(s) WHERE s.name =~ '(?i).*\\b(python|machine learning|data|ai)\\b.*' | s]) * 1 +
             size([(f)-[w:WORKED_AT]->() WHERE w.role =~ '(?i).*\\b(ai|data|machine learning)\\b.*' | w]) * 2 +
             size([(f)-[:CREATED_PROJECT]->(p) WHERE p.name =~ '(?i).*\\b(ai|model)\\b.*' | p]) * 1 AS ai_score,

             size([(f)-[:HAS_SKILL]->(s) WHERE s.name =~ '(?i).*\\b(ui|ux|design|figma)\\b.*' | s]) * 1 +
             size([(f)-[w:WORKED_AT]->() WHERE w.role =~ '(?i).*\\b(design|ui|ux)\\b.*' | w]) * 2 +
             size([(f)-[:CREATED_PROJECT]->(p) WHERE p.name =~ '(?i).*\\b(design|interface)\\b.*' | p]) * 1 AS ux_score,

             size([(f)-[:HAS_SKILL]->(s) WHERE s.name =~ '(?i).*\\b(web|react|javascript|html|node)\\b.*' | s]) * 1 +
             size([(f)-[w:WORKED_AT]->() WHERE w.role =~ '(?i).*\\b(web|developer|front|back)\\b.*' | w]) * 2 +
             size([(f)-[:CREATED_PROJECT]->(p) WHERE p.name =~ '(?i).*\\b(web|app)\\b.*' | p]) * 1 AS web_score

        // 2. COLLECT AND PERMUTE CANDIDATES
        WITH collect({person: f, ai: ai_score, ux: ux_score, web: web_score}) AS candidates
        UNWIND candidates AS c1
        UNWIND candidates AS c2
        UNWIND candidates AS c3

        // 3. FILTER FOR DISTINCT PEOPLE WITH AT LEAST 1 POINT IN THEIR ROLE
        WITH c1, c2, c3
        WHERE c1.person <> c2.person
          AND c2.person <> c3.person
          AND c1.person <> c3.person
          AND c1.ai > 0
          AND c2.ux > 0
          AND c3.web > 0

        WITH c1.person AS f1, c1.ai AS ai_score,
             c2.person AS f2, c2.ux AS ux_score,
             c3.person AS f3, c3.web AS web_score

        // 4. CALCULATE SYNERGY
        WITH f1, ai_score, f2, ux_score, f3, web_score,
             size([(f1)-[:HAS_SKILL|STUDIED_AT|WORKED_AT|CREATED_PROJECT]->(shared)<-[:HAS_SKILL|STUDIED_AT|WORKED_AT|CREATED_PROJECT]-(f2) | shared]) AS overlap_12,
             size([(f2)-[:HAS_SKILL|STUDIED_AT|WORKED_AT|CREATED_PROJECT]->(shared)<-[:HAS_SKILL|STUDIED_AT|WORKED_AT|CREATED_PROJECT]-(f3) | shared]) AS overlap_23,
             size([(f1)-[:HAS_SKILL|STUDIED_AT|WORKED_AT|CREATED_PROJECT]->(shared)<-[:HAS_SKILL|STUDIED_AT|WORKED_AT|CREATED_PROJECT]-(f3) | shared]) AS overlap_13

        WITH f1, f2, f3, ai_score, ux_score, web_score,
             (overlap_12 + overlap_23 + overlap_13) AS synergy_score,
             (ai_score + ux_score + web_score) AS total_expertise_score

        // 5. THE DECIMAL TIE-BREAKER LOGIC
        RETURN f1.name AS AI_Engineer, ai_score AS AI_Suitability,
               f2.name AS UI_UX_Designer, ux_score AS UX_Suitability,
               f3.name AS Web_Developer, web_score AS Web_Suitability,
               synergy_score AS Team_Synergy,
               total_expertise_score AS Total_Tech_Score,
               (total_expertise_score + (synergy_score * 0.01)) AS Final_Team_Rating
        ORDER BY Final_Team_Rating DESC
        LIMIT $limit
        """
        
        with self.driver.session() as session:
            # We pass the limit as a parameter
            result = session.run(query, limit=limit)
            teams = [record.data() for record in result]
            
            return teams


In [6]:
# --- Usage Example to put at the bottom of your script ---
if __name__ == "__main__":
    kg = KnowledgeGraphReader()
    
    print("\n🔍 Generating Optimal Team Formations...\n")
    top_teams = kg.form_optimal_teams(limit=5)
    
    if not top_teams:
        print("❌ No valid teams could be formed. (Missing required roles in the database)")
    else:
        for i, team in enumerate(top_teams, 1):
            print(f"🏆 TEAM #{i} | Rating: {team['Final_Team_Rating']:.2f} (Tech: {team['Total_Tech_Score']}, Synergy: {team['Team_Synergy']})")
            print(f"   🤖 AI  : {team['AI_Engineer']} (Score: {team['AI_Suitability']})")
            print(f"   🎨 UX  : {team['UI_UX_Designer']} (Score: {team['UX_Suitability']})")
            print(f"   💻 WEB : {team['Web_Developer']} (Score: {team['Web_Suitability']})")
            print("-" * 65)

    kg.close()

✅ Successfully connected to the existing Knowledge Graph!

🔍 Generating Optimal Team Formations...

🏆 TEAM #1 | Rating: 19.25 (Tech: 19, Synergy: 25)
   🤖 AI  : Youssef Maaod (Score: 8)
   🎨 UX  : Samaa Khaled Mohamed (Score: 3)
   💻 WEB : ADHAM MOHMED ELWAKEEL (Score: 8)
-----------------------------------------------------------------
🏆 TEAM #2 | Rating: 19.23 (Tech: 19, Synergy: 23)
   🤖 AI  : Habiba Ahmed AbdElnapy (Score: 8)
   🎨 UX  : Samaa Khaled Mohamed (Score: 3)
   💻 WEB : ADHAM MOHMED ELWAKEEL (Score: 8)
-----------------------------------------------------------------
🏆 TEAM #3 | Rating: 16.23 (Tech: 16, Synergy: 23)
   🤖 AI  : Habiba Ahmed AbdElnapy (Score: 8)
   🎨 UX  : Samaa Khaled Mohamed (Score: 3)
   💻 WEB : Hamed Emad Hamed Farrag (Score: 5)
-----------------------------------------------------------------
🏆 TEAM #4 | Rating: 16.23 (Tech: 16, Synergy: 23)
   🤖 AI  : Youssef Maaod (Score: 8)
   🎨 UX  : Samaa Khaled Mohamed (Score: 3)
   💻 WEB : Hamed Emad Hamed Farrag